##Load and inspect Bronze patient encounters

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "capstone_8hospital"
BRONZE = f"{CATALOG}.bronze_schema"
SILVER = f"{CATALOG}.silver_schema"
encounters_bronze = spark.table(f"{BRONZE}.bronze_patient_encounters")
display(encounters_bronze.limit(10))
print("Total encounter rows:", encounters_bronze.count())

encounter_id,patient_id,department_id,admission_ts,discharge_ts,encounter_type,triage_priority,encounter_status,attending_clinician_id,admission_source,diagnosis_group,bed_id,_rescued_data,source_file,ingestion_timestamp
ENCOUNT080003715,PATIEN0800003,DEPART0800152,2026-07-19T05:48:00Z,2026-07-19T05:53:00Z,INPATIENT,P1,ADMITTED,ATTENDI080003715,ADMIS_6,DIAGN_6,BED0800167,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_004.csv,2026-11-04T06:08:09.792Z
ENCOUNT080003716,PATIEN0800141,DEPART0800023,2026-07-19T05:55:00Z,2026-07-19T06:00:00Z,OUTPATIENT,P1,TRANSFERRED,ATTENDI080003716,ADMIS_7,DIAGN_7,BED0800182,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_004.csv,2026-11-04T06:08:09.792Z
ENCOUNT080003717,PATIEN0800005,DEPART0800085,2026-07-19T06:02:00Z,2026-07-19T06:07:00Z,EMERGENCY,P1,DISCHARGED,ATTENDI080003717,ADMIS_1,DIAGN_1,BED0800195,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_004.csv,2026-11-04T06:08:09.792Z
ENCOUNT080003718,PATIEN0800177,DEPART0800079,2026-07-19T06:09:00Z,2026-07-19T06:14:00Z,EMERGENCY,P2,TRANSFERRED,ATTENDI080003718,ADMIS_2,DIAGN_2,BED0800024,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_004.csv,2026-11-04T06:08:09.792Z
ENCOUNT080003719,PATIEN0800043,DEPART0800050,2026-07-19T06:16:00Z,2026-07-19T06:21:00Z,INPATIENT,P3,TRANSFERRED,ATTENDI080003719,ADMIS_3,DIAGN_3,BED0800039,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_004.csv,2026-11-04T06:08:09.792Z
ENCOUNT080003720,PATIEN0800179,DEPART0800050,2026-07-19T06:23:00Z,2026-07-19T06:28:00Z,INPATIENT,P4,TRANSFERRED,ATTENDI080003720,ADMIS_4,DIAGN_4,BED0800102,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_004.csv,2026-11-04T06:08:09.792Z
ENCOUNT080003721,PATIEN0800144,DEPART0800095,2026-07-19T06:30:00Z,2026-07-19T06:35:00Z,EMERGENCY,P1,ADMITTED,ATTENDI080003721,ADMIS_5,DIAGN_5,BED0800028,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_004.csv,2026-11-04T06:08:09.792Z
ENCOUNT080003722,PATIEN0800149,DEPART0800101,2026-07-19T06:37:00Z,2026-07-19T06:42:00Z,OUTPATIENT,P1,TRANSFERRED,ATTENDI080003722,ADMIS_6,DIAGN_6,BED0800164,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_004.csv,2026-11-04T06:08:09.792Z
ENCOUNT080003723,PATIEN0800062,DEPART0800069,2026-07-19T06:44:00Z,2026-07-19T06:49:00Z,INPATIENT,P1,DISCHARGED,ATTENDI080003723,ADMIS_7,DIAGN_7,BED0800088,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_004.csv,2026-11-04T06:08:09.792Z
ENCOUNT080003724,PATIEN0800027,DEPART0800089,2026-07-19T06:51:00Z,2026-07-19T06:56:00Z,OUTPATIENT,P1,DISCHARGED,ATTENDI080003724,ADMIS_1,DIAGN_1,BED0800049,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_004.csv,2026-11-04T06:08:09.792Z


Total encounter rows: 12376


##Clean patient encounter columns

In [0]:
encounters_checked = (
    encounters_bronze
    .withColumn("encounter_id", F.trim("encounter_id"))
    .withColumn("patient_id", F.trim("patient_id"))
    .withColumn("department_id", F.trim("department_id"))
    .withColumn("bed_id", F.trim("bed_id"))
    .withColumn("admission_ts", F.to_timestamp("admission_ts"))
    .withColumn("discharge_ts", F.to_timestamp("discharge_ts"))
    .withColumn("encounter_type", F.upper(F.trim("encounter_type")))
    .withColumn("triage_priority", F.upper(F.trim("triage_priority")))
    .withColumn("encounter_status", F.upper(F.trim("encounter_status")))
    .withColumn("attending_clinician_id", F.trim("attending_clinician_id"))
    .withColumn("admission_source", F.upper(F.trim("admission_source")))
    .withColumn("diagnosis_group", F.upper(F.trim("diagnosis_group")))
)

##Check mandatory fields and timestamp rules

In [0]:
encounters_checked = encounters_checked.withColumn(
    "dq_reason",
    F.concat_ws(", ",
        F.when(F.col("encounter_id").isNull() | (F.col("encounter_id") == ""), "MISSING_ENCOUNTER_ID"),
        F.when(F.col("patient_id").isNull() | (F.col("patient_id") == ""), "MISSING_PATIENT_ID"),
        F.when(F.col("department_id").isNull() | (F.col("department_id") == ""), "MISSING_DEPARTMENT_ID"),
        F.when(F.col("admission_ts").isNull(), "INVALID_ADMISSION_TIMESTAMP"),
        F.when(F.col("discharge_ts").isNotNull() & (F.col("discharge_ts") < F.col("admission_ts")), "DISCHARGE_BEFORE_ADMISSION")
    )
)

encounters_quarantine = encounters_checked.filter(F.col("dq_reason") != "")
encounters_valid = encounters_checked.filter(F.col("dq_reason") == "")

##Validate patient and department references

In [0]:
patient_keys = spark.table(f"{SILVER}.silver_patients").select("patient_id").distinct()
department_keys = spark.table(f"{SILVER}.silver_departments").select("department_id").distinct()
bed_keys = spark.table(f"{SILVER}.silver_beds").select("bed_id").distinct()

encounters_valid = (
    encounters_valid
    .join(patient_keys.withColumn("patient_found", F.lit(1)), "patient_id", "left")
    .join(department_keys.withColumn("department_found", F.lit(1)), "department_id", "left")
    .join(bed_keys.withColumn("bed_found", F.lit(1)), "bed_id", "left")
    .withColumn("dq_reason", F.concat_ws(", ",
        F.when(F.col("patient_found").isNull(), "UNKNOWN_PATIENT_ID"),
        F.when(F.col("department_found").isNull(), "UNKNOWN_DEPARTMENT_ID"),
        F.when(F.col("bed_id").isNotNull() & F.col("bed_found").isNull(), "UNKNOWN_BED_ID")
    ))
)

encounters_quarantine = encounters_quarantine.unionByName(
    encounters_valid.filter(F.col("dq_reason") != ""), allowMissingColumns=True
)
encounters_valid = encounters_valid.filter(F.col("dq_reason") == "")

##Deduplicate patient encounters

In [0]:
encounter_window = Window.partitionBy("encounter_id").orderBy(
    F.col("ingestion_timestamp").desc_nulls_last(),
    F.col("source_file").asc_nulls_last()
)

encounters_ranked = encounters_valid.withColumn("row_num", F.row_number().over(encounter_window))
encounters_duplicate_audit = encounters_ranked.filter(F.col("row_num") > 1)
encounters_silver = encounters_ranked.filter(F.col("row_num") == 1).drop(
    "row_num", "dq_reason", "patient_found", "department_found", "bed_found"
)

##Save Patient Encounters Silver and Quarantine

In [0]:
(encounters_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.silver_patient_encounters"))

(encounters_quarantine.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.silver_encounter_quarantine"))

(encounters_duplicate_audit.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.silver_encounter_duplicate_audit"))

##Validate saved Encounter tables

In [0]:
for table_name in ["silver_patient_encounters", "silver_encounter_quarantine", "silver_encounter_duplicate_audit"]:
    print(table_name, spark.table(f"{SILVER}.{table_name}").count())

display(spark.table(f"{SILVER}.silver_patient_encounters").limit(10))

silver_patient_encounters 12329
silver_encounter_quarantine 0
silver_encounter_duplicate_audit 47


bed_id,department_id,patient_id,encounter_id,admission_ts,discharge_ts,encounter_type,triage_priority,encounter_status,attending_clinician_id,admission_source,diagnosis_group,_rescued_data,source_file,ingestion_timestamp
BED0800175,DEPART0800191,PATIEN0800037,ENCOUNT080000001,2026-07-01T04:30:00.000Z,2026-07-01T04:35:00.000Z,INPATIENT,P3,DISCHARGED,ATTENDI080000001,ADMIS_2,DIAGN_2,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_001.csv,2026-11-04T06:08:09.792Z
BED0800159,DEPART0800136,PATIEN0800093,ENCOUNT080000002,2026-07-01T04:37:00.000Z,2026-07-01T04:42:00.000Z,INPATIENT,P3,TRANSFERRED,ATTENDI080000002,ADMIS_3,DIAGN_3,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_001.csv,2026-11-04T06:08:09.792Z
BED0800197,DEPART0800127,PATIEN0800040,ENCOUNT080000003,2026-07-01T04:44:00.000Z,2026-07-01T04:49:00.000Z,EMERGENCY,P2,ADMITTED,ATTENDI080000003,ADMIS_4,DIAGN_4,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_001.csv,2026-11-04T06:08:09.792Z
BED0800125,DEPART0800098,PATIEN0800117,ENCOUNT080000004,2026-07-01T04:51:00.000Z,2026-07-01T04:56:00.000Z,INPATIENT,P3,ADMITTED,ATTENDI080000004,ADMIS_5,DIAGN_5,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_001.csv,2026-11-04T06:08:09.792Z
BED0800064,DEPART0800112,PATIEN0800095,ENCOUNT080000005,2026-07-01T04:58:00.000Z,2026-07-01T05:03:00.000Z,OUTPATIENT,P4,TRANSFERRED,ATTENDI080000005,ADMIS_6,DIAGN_6,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_001.csv,2026-11-04T06:08:09.792Z
BED0800136,DEPART0800122,PATIEN0800015,ENCOUNT080000006,2026-07-01T05:05:00.000Z,2026-07-01T05:10:00.000Z,OUTPATIENT,P3,DISCHARGED,ATTENDI080000006,ADMIS_7,DIAGN_7,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_001.csv,2026-11-04T06:08:09.792Z
BED0800171,DEPART0800037,PATIEN0800123,ENCOUNT080000007,2026-07-01T05:12:00.000Z,2026-07-01T05:17:00.000Z,OUTPATIENT,P2,TRANSFERRED,ATTENDI080000007,ADMIS_1,DIAGN_1,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_001.csv,2026-11-04T06:08:09.792Z
BED0800199,DEPART0800112,PATIEN0800124,ENCOUNT080000008,2026-07-01T05:19:00.000Z,2026-07-01T05:24:00.000Z,OUTPATIENT,P4,DISCHARGED,ATTENDI080000008,ADMIS_2,DIAGN_2,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_001.csv,2026-11-04T06:08:09.792Z
BED0800164,DEPART0800101,PATIEN0800028,ENCOUNT080000009,2026-07-01T05:26:00.000Z,2026-07-01T05:31:00.000Z,OUTPATIENT,P2,DISCHARGED,ATTENDI080000009,ADMIS_3,DIAGN_3,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_001.csv,2026-11-04T06:08:09.792Z
BED0800067,DEPART0800141,PATIEN0800169,ENCOUNT080000010,2026-07-01T05:33:00.000Z,2026-07-01T05:38:00.000Z,EMERGENCY,P1,TRANSFERRED,ATTENDI080000010,ADMIS_4,DIAGN_4,null,/Volumes/capstone_8hospital/bronze_schema/ext_capstone8_vol/patient_encounters/patient_encounters_batch_001.csv,2026-11-04T06:08:09.792Z
